# [1장 통합 실습] 사내 LLM의 출력 점수와 손실 계산

세 TODO를 채우고 결과를 확인하세요.

## 해결할 상황

회사에서 사용하는 Private LLM 도우미에 평가 화면을 붙이고 있습니다. 화면에는 문의를 어느 부서로 보낼지, 작성된 답변의 품질이 어느 정도인지, 답변에 민감한 정보가 포함됐을 가능성이 있는지가 함께 표시됩니다. 그런데 부서별 점수를 확률로 바꾸는 과정에서 숫자가 깨지고, 세 출력을 전부 같은 방식으로 채점하려던 코드도 아직 완성되지 않았습니다.

여러분은 **출력 형태에 맞는 계산을 구현하고, 평균 점수 뒤에 가려진 잘못된 예측 한 건을 찾아 설명**해야 합니다. 모델 호출이나 학습은 하지 않습니다. 문의, 점수, 정답은 모두 수업용 가상 자료이며 계산 자체는 NumPy로 실행합니다.

## 자료를 읽는 방법

배열의 같은 행은 항상 같은 문의입니다. A01은 메일 접속 장애, A02는 연차 신청, A03은 비밀 문서 외부 노출, A04는 노트북 도난 문의입니다. `TEXTS`는 문의 종류를 알아보기 위한 짧은 이름이며, 이 이름에서 모델 출력을 새로 만드는 코드는 없습니다.

| 평가 항목 | 제공 예측 | 제공 정답 | 계산할 손실 |
| --- | --- | --- | --- |
| 담당 부서 하나 선택 | `LOGITS`: `(4, 3)` 원시 점수 | `TARGETS`: `(4,)` 클래스 번호 | Cross Entropy, 줄여서 CE |
| 답변 품질 예측 | `QUALITY_PRED`: `(4,)` 연속 점수 | `QUALITY_GOLD`: `(4,)` 검수 점수 | MSE |
| 민감정보 포함 여부 | `SENSITIVE_PROB`: `(4,)` 확률 | `SENSITIVE_GOLD`: `(4,)`의 0/1 | BCE |

부서 번호는 `0=it`, `1=hr`, `2=security`입니다. 품질 점수는 0~5 범위이고, 민감정보 정답 1은 해당 가상 답변에 민감한 정보가 포함됐다는 뜻입니다. 문의 종류만으로 이 정답을 새로 추론하지 않고 제공된 검수값을 사용합니다.

`LOGITS`의 1000 이상 값은 확률이 아닙니다. 각 행에서 부서 간 점수 차이가 중요합니다. BCE에 전달하는 `SENSITIVE_PROB`는 이미 확률이므로 부서 logits와 섞어서 사용하지 않습니다.

## 작성할 세 부분

시작 코드의 TODO 세 묶음을 완성하세요. 함수 안의 TODO는 공백 네 칸 들여쓰기를 유지합니다. TODO 3은 함수 밖에서 실행하는 계산입니다. 제공된 데이터와 마지막 출력·검사 코드는 그대로 사용합니다.

### TODO 1. 큰 점수를 안정적인 확률로 바꾸기

`distribution(logits)`의 입력은 `(문의 수, 부서 수)` 모양의 NumPy 배열입니다. 반환값은 **확률 배열, log 확률 배열** 순서이며 두 배열의 shape은 입력과 같아야 합니다.

각 문의의 부서 점수에 대해 다음 계산을 구현하세요.

1. 마지막 축의 최댓값을 구하고 각 점수에서 뺍니다.
2. 이동한 점수를 지수화하고, 같은 마지막 축의 합으로 나눠 확률을 만듭니다.
3. log 확률은 **이동한 점수 − log(지수값의 합)**으로 계산합니다. 완성된 확률에 단순히 log를 취하는 경로는 사용하지 않습니다.
4. 확률과 log 확률을 순서대로 반환합니다.

`np.max`, `np.exp`, `np.sum`, `np.log`를 사용할 수 있습니다. 부서가 마지막 축이므로 `axis=-1`입니다. 최댓값과 합계를 구할 때 `keepdims=True`를 사용하면 `(4, 1)`이 유지되어 각 행에 맞게 빼거나 나눌 수 있습니다.

행 전체에 같은 상수를 더하거나 빼도 softmax 확률은 같아야 합니다. 최댓값을 빼면 가장 큰 지수값이 1이 되어 큰 수의 지수 계산을 피할 수 있습니다. 이 계산은 점수의 크기를 안정화하는 것이며 예측이 옳다는 보장은 아닙니다.

### TODO 2. 세 출력에 맞는 문의별 손실 구하기

`sample_losses`는 위에서 얻은 log 확률과 정답, 품질 예측과 정답, 민감정보 확률과 정답을 받습니다. 반환값은 **`ce_each`, `mse_each`, `bce_each`** 순서이며 각각 길이 4인 배열이어야 합니다. 여기서는 평균내지 않습니다.

- **CE:** 각 행에서 정답 부서의 log 확률 하나를 고르고 부호를 바꿉니다. `np.arange`로 문의별 행 번호를 만들고, 행 번호와 `targets`를 함께 사용해 원소를 고를 수 있습니다.
- **MSE:** 품질 예측에서 정답을 뺀 오차를 제곱합니다. 문의별 제곱 오차를 유지합니다.
- **BCE:** 민감정보 정답을 `y`, 확률을 `p`라고 하면 문의별 값은 `−[y·log(p)+(1−y)·log(1−p)]`입니다. log 계산 전에 확률을 `1e-12`와 `1-1e-12` 사이로 제한하세요. `np.clip`을 사용할 수 있습니다.

확률 0과 1도 입력에 들어 있습니다. clip은 log(0)을 피하기 위한 수치 처리이며, 예측이 틀렸는데 정답으로 바꾸는 처리가 아닙니다. 정답 배열은 모두 해당 예측 배열과 같은 문의 순서입니다.

### TODO 3. 평균과 문제가 큰 문의를 함께 찾기

함수 실행 후 준비된 `probs`, `ce_each`, `mse_each`, `bce_each`를 사용합니다. 다음 변수를 완성하세요.

| 변수 | 필요한 값 |
| --- | --- |
| `predicted` | 각 문의에서 확률이 가장 큰 부서 번호, shape `(4,)` |
| `correct` | 각 예측 번호가 `TARGETS`와 같은지 나타내는 True/False 배열 |
| `ce_mean`, `mse_mean`, `bce_mean` | 각각 문의별 손실의 산술평균 |
| `worst` | CE가 가장 큰 문의의 행 번호 하나 |
| `correct_count` | 맞힌 문의 수를 나타내는 정수 |

`np.argmax`, `np.mean`, `np.sum`을 사용할 수 있습니다. 부서를 고를 때는 각 행의 마지막 축에서 최대 위치를 찾아야 합니다. 평균을 구한 뒤에는 이미 문의별 구분이 사라지므로, `worst`는 평균이 아니라 `ce_each`에서 찾습니다.

마지막 제공 코드는 각 문의의 세 손실과 전체 결과를 출력합니다.

## 패키지 설치

설치 셀부터 순서대로 실행합니다.

In [ ]:
%pip install -q "numpy==2.4.6"


## 1. 제공 자료

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
# 동일한 사내 문의 네 건에 대한 가상 출력입니다. 모델을 호출하거나 학습하지 않습니다.
IDS = ("A01", "A02", "A03", "A04")
TEXTS = ("메일 접속 장애", "연차 신청", "비밀 문서 외부 노출", "노트북 도난")
LABELS = ("it", "hr", "security")
# 마지막 축은 담당 부서입니다. 큰 값도 확률이 아니라 정규화 전 점수입니다.
LOGITS = np.array([[1002., 1000., 999.], [1000., 1002., 999.],
                   [1001., 1000., 1003.], [1006., 1000., 999.]])
TARGETS = np.array([0, 1, 2, 2])  # 각 문의의 정답 클래스 번호
# 답변 품질은 0~5점 연속값, 민감정보 포함 여부는 0/1입니다.
QUALITY_PRED = np.array([4.5, 4.0, 2.5, 4.0])
QUALITY_GOLD = np.array([4.0, 5.0, 2.0, 1.0])
SENSITIVE_PROB = np.array([0.0, 1.0, 0.6, 0.01])
SENSITIVE_GOLD = np.array([0.0, 1.0, 1.0, 1.0])




## 2. 직접 구현할 계산

In [ ]:
def distribution(logits):
    """[문의, 부서] logits를 같은 shape의 확률과 log 확률로 바꿉니다."""
    # TODO 1: 문의별 클래스 축을 따라 안정적인 두 분포를 계산합니다.
    raise NotImplementedError("TODO 1: 위 요구사항에 맞게 구현하세요.")


def sample_losses(log_probs, targets, quality_pred, quality_gold, flag_prob, flag_gold):
    """문의별 CE, MSE, BCE를 각각 길이 N인 배열로 반환합니다."""
    # TODO 2: 출력 형태에 맞는 손실을 계산합니다. 아직 평균내지 않습니다.
    raise NotImplementedError("TODO 2: 위 요구사항에 맞게 구현하세요.")




## 3. 실행과 결과 비교

In [ ]:
probs, log_probs = distribution(LOGITS)
ce_each, mse_each, bce_each = sample_losses(
    log_probs, TARGETS, QUALITY_PRED, QUALITY_GOLD, SENSITIVE_PROB, SENSITIVE_GOLD,
)
# TODO 3: 예측 부서, 정답 여부, 평균 손실과 가장 큰 CE의 행 번호를 구합니다.
raise NotImplementedError("TODO 3: 위 요구사항에 맞게 구현하세요.")

# 제공 검증: 유한성·행별 정규화·상수 이동 불변성을 함께 확인합니다.
assert probs.shape == LOGITS.shape and log_probs.shape == LOGITS.shape
assert np.isfinite(probs).all() and np.isfinite(log_probs).all()
assert np.allclose(probs.sum(axis=-1), 1.0)
assert np.allclose(np.exp(log_probs), probs)
assert np.allclose(distribution(LOGITS - 1000.0)[0], probs)
assert np.isfinite(bce_each).all()
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(LOGITS) / np.exp(LOGITS).sum(axis=-1, keepdims=True)
print("단순 exp 계산은 모두 유한한가:", np.isfinite(naive).all())
print("안정적인 확률:\n", probs)
for i, sample_id in enumerate(IDS):
    print(f"{sample_id} {TEXTS[i]} | 예측={LABELS[predicted[i]]}, 정답={LABELS[TARGETS[i]]}"
          f" | CE={ce_each[i]:.4f}, MSE={mse_each[i]:.4f}, BCE={bce_each[i]:.4f}")
print(f"정답 수: {correct_count}/{len(TARGETS)}")
print(f"평균 CE={ce_mean:.4f}, MSE={mse_mean:.4f}, BCE={bce_mean:.4f}")
print("가장 큰 CE:", IDS[worst], "| 정답 부서 확률:", round(probs[worst, TARGETS[worst]], 6))


## 실행 후 확인

완성한 코드를 처음부터 실행하고 다음 결과가 나오는지 확인하세요. 소수점 표시 자릿수에 따른 작은 차이는 괜찮습니다.

| 확인 항목 | 기대 결과 |
| --- | --- |
| 단순 exp 계산은 모두 유한한가 | `False` |
| 안정적으로 계산한 각 행의 확률 합 | 모두 1 |
| 예측 부서 번호 | 0, 1, 2, 0 |
| 맞힌 문의 수 | 3/4 |
| 평균 CE / MSE / BCE | 1.8782 / 2.6250 / 1.2790 |
| CE가 가장 큰 문의 | A04 |

`False`는 제공된 단순 계산 예시가 큰 logits를 처리하지 못했다는 뜻입니다. 직접 구현한 `probs`, `log_probs`의 유한성 검사는 통과해야 합니다. 출력만 맞추기 위해 값을 직접 입력하지 말고 제공 배열에서 계산하세요.

같은 결과 아래에 다음 세 내용을 짧게 적으면 이 문의 평가 작업이 마무리됩니다.

1. A04에서 **예측한 부서와 그 확률, 정답 부서와 그 확률, CE**를 확인하세요. 확률 계산이 안정적이어도 정답을 틀릴 수 있는 이유를 설명하세요.
2. 담당 부서, 답변 품질, 민감정보 여부에 서로 다른 손실을 사용한 이유를 각각 한 문장으로 적으세요. 평균을 구하기 전에 문의별 손실을 남겨 둔 이유도 덧붙이세요.
3. logits에서 1000을 뺀 결과가 같은 이유와 BCE의 0·1 확률을 clip한 이유를 구분해 설명하세요. 두 처리가 예측의 정답 여부를 바꾸는지도 적으세요.

## 나의 관찰 메모

위의 관찰 질문에 대한 답을 이 텍스트 셀에 작성하세요.
